In [4]:
#!/usr/bin/env python3
"""
Iris Flower Classification
==========================
Classifies an iris flower as setosa, versicolor or virginica from its four measurements
(sepal length / width, petal length / width).

Pipeline
--------
1. Load + validate : checks columns, missing values, duplicates, class balance.
2. Explore         : per-species statistics and a pair plot.
3. Split           : 80/20 stratified hold-out. The test flowers are NOT touched until the end.
4. Compare models  : Logistic Regression, KNN, SVM, Decision Tree, Random Forest, Naive Bayes,
                     using 10x repeated 5-fold cross-validation on the TRAINING part only
                     (150 flowers is small, so repeating gives a more stable ranking).
5. Evaluate        : the winner is scored once on the hold-out test flowers, and also with
                     5-fold cross-validated predictions over all 150 flowers (more stable,
                     because 30 test flowers means each mistake costs 3.3 points).
6. Save            : final model trained on all 150 flowers -> iris_model.joblib, charts, report.

Usage
-----
    python iris_classification.py                       # finds IRIS.csv next to the script
    python iris_classification.py --csv path/to/IRIS.csv
    python iris_classification.py --predict 5.9 3.0 5.1 1.8   # classify one new flower
Works as a normal script and when pasted into one Google Colab / Jupyter cell
(upload IRIS.csv to the working folder first).
"""
import argparse
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_predict, cross_validate, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

SEED = 42
OUT_DIR = "iris_output"
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
SPECIES = ["setosa", "versicolor", "virginica"]
CODE = {s: i for i, s in enumerate(SPECIES)}

_LOG = []


def log(text=""):
    """print + remember, so the same text is saved to results_summary.txt"""
    print(text)
    _LOG.append(str(text))


# --------------------------------------------------------------------------- #
# 1. Load + validate
# --------------------------------------------------------------------------- #
def find_local_csv():
    # __file__ does not exist inside a Colab / Jupyter cell -> fall back to the working folder
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    for folder in (here, os.path.join(here, "data"), os.getcwd(), os.path.join(os.getcwd(), "data")):
        for name in ("IRIS.csv", "Iris.csv", "iris.csv", "IRIS[1].csv", "Iris[1].csv", "iris[1].csv"):
            p = os.path.join(folder, name)
            if os.path.isfile(p):
                return p
    return None


def load_data(csv_path=None):
    csv_path = csv_path or find_local_csv()
    if csv_path is None:
        sys.exit("Could not find IRIS.csv. Put it next to this script (or in a 'data' folder) "
                 "or run:  python iris_classification.py --csv <path>")
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    missing = set(FEATURES + ["species"]) - set(df.columns)
    if missing:
        sys.exit(f"Unexpected file format, missing columns: {sorted(missing)}")
    df = df[FEATURES + ["species"]].copy()
    df["species"] = df["species"].astype(str).str.strip().str.replace(r"^Iris-", "", regex=True)
    n_missing = int(df.isna().sum().sum())
    df = df.dropna().reset_index(drop=True)
    unknown = set(df["species"]) - set(SPECIES)
    if unknown:
        sys.exit(f"Unexpected species labels: {sorted(unknown)}")
    log(f"Loaded {len(df)} flowers from {csv_path}")
    log(f"Missing values dropped: {n_missing}.  Exact duplicate rows kept: {int(df.duplicated().sum())} "
        "(real flowers can share measurements).")
    return df


# --------------------------------------------------------------------------- #
# 2. Plot style (3 species = 3 categorical hues + 3 marker shapes, so colour is never the only cue)
# --------------------------------------------------------------------------- #
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e6e5e0"
BLUE = "#2a78d6"
COLORS = {"setosa": "#2a78d6", "versicolor": "#eb6834", "virginica": "#1baf7a"}
MARKERS = {"setosa": "o", "versicolor": "s", "virginica": "^"}


def style():
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
        "xtick.color": INK2, "ytick.color": INK2, "axes.titlecolor": INK,
        "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlesize": 12,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
        "axes.axisbelow": True, "font.size": 10,
    })


def save(fig, out, name, rect=None):
    fig.tight_layout(rect=rect) if rect else fig.tight_layout()
    fig.savefig(os.path.join(out, name), dpi=150)
    plt.close(fig)


def species_legend():
    return [Line2D([0], [0], marker=MARKERS[s], color="none", markerfacecolor=COLORS[s],
                   markeredgecolor="none", markersize=8, label=s) for s in SPECIES]


# --------------------------------------------------------------------------- #
# 3. Exploration
# --------------------------------------------------------------------------- #
def explore(df, out):
    log("\n=== Data overview ===")
    log("Flowers per species: " + ", ".join(f"{s} {n}" for s, n in df["species"].value_counts().items()))
    log("\nAverage measurements (cm) per species:")
    log(df.groupby("species")[FEATURES].mean().round(2).to_string())
    pl = df.groupby("species")["petal_length"].agg(["min", "max"])
    log("\nPetal length range (cm) per species:")
    log(pl.to_string())
    if pl.loc["setosa", "max"] < pl.loc["versicolor", "min"]:
        log(f"-> Setosa is completely separable: its largest petal ({pl.loc['setosa', 'max']} cm) "
            f"is shorter than the smallest versicolor petal ({pl.loc['versicolor', 'min']} cm).")
    ov = pl.loc["versicolor", "max"] - pl.loc["virginica", "min"]
    if ov > 0:
        log(f"-> Versicolor and virginica overlap by {ov:.1f} cm of petal length, "
            "so that pair is where mistakes can happen.")

    style()
    # same axis range for a feature in every panel, so gridlines line up across the grid
    lims = {f: (df[f].min() - 0.1 * (df[f].max() - df[f].min()),
                df[f].max() + 0.1 * (df[f].max() - df[f].min())) for f in FEATURES}
    fig, axes = plt.subplots(4, 4, figsize=(9.5, 9.5))
    for i, fy in enumerate(FEATURES):
        left_cell = 0 if i != 0 else 1  # first off-diagonal panel in each row carries the y-axis
        for j, fx in enumerate(FEATURES):
            ax = axes[i, j]
            for sp in SPECIES:
                s = df[df["species"] == sp]
                if i == j:
                    ax.hist(s[fx], bins=12, color=COLORS[sp], alpha=0.6, edgecolor=SURFACE)
                else:
                    ax.scatter(s[fx], s[fy], s=16, color=COLORS[sp], marker=MARKERS[sp],
                               alpha=0.75, edgecolor="none")
            ax.set_xlim(*lims[fx])
            if i == j:  # diagonal = histogram of counts: no y-axis, just name the feature
                ax.set_yticks([])
                ax.set_title(fx.replace("_", " "), loc="center", fontsize=10)
            else:
                ax.set_ylim(*lims[fy])
                if j == left_cell:
                    ax.set_ylabel(fy.replace("_", " ") + " (cm)")
                else:
                    ax.tick_params(labelleft=False)
            if i == 3:
                ax.set_xlabel(fx.replace("_", " ") + " (cm)")
            else:
                ax.tick_params(labelbottom=False)
    fig.legend(handles=species_legend(), loc="upper right", ncol=3, frameon=False)
    fig.suptitle("Petal measurements separate the species best", x=0.01, ha="left",
                 fontweight="bold", fontsize=13)
    save(fig, out, "pairplot.png", rect=[0, 0, 1, 0.95])


# --------------------------------------------------------------------------- #
# 4. Models
# --------------------------------------------------------------------------- #
def make_models():
    scaled = lambda m: Pipeline([("scale", StandardScaler()), ("model", m)])
    return {
        "Logistic Regression": scaled(LogisticRegression(max_iter=1000, random_state=SEED)),
        "K-Nearest Neighbors": scaled(KNeighborsClassifier(n_neighbors=7)),
        "SVM (RBF kernel)": scaled(SVC(C=1.0, probability=True, random_state=SEED)),
        "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
        "Naive Bayes": GaussianNB(),
    }


def compare_models(X, y, out):
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
    rows = []
    for name, model in make_models().items():
        res = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "f1_macro"])
        rows.append({"model": name, "accuracy": res["test_accuracy"].mean(),
                     "accuracy_std": res["test_accuracy"].std(),
                     "f1_macro": res["test_f1_macro"].mean()})
    res = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
    res.round(4).to_csv(os.path.join(out, "model_comparison.csv"), index=False)

    style()
    r = res.iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    ax.errorbar(r["accuracy"], np.arange(len(r)), xerr=r["accuracy_std"], fmt="o", color=BLUE,
                ecolor=BLUE, elinewidth=1.5, capsize=3, markersize=8, markeredgecolor=SURFACE)
    for y_, a in enumerate(r["accuracy"]):  # value column at the right edge, aligned with each dot
        ax.text(1.02, y_, f"{a:.1%}", transform=ax.get_yaxis_transform(), va="center",
                fontsize=9, color=INK)
    ax.set_yticks(range(len(r)))
    ax.set_yticklabels(r["model"])
    ax.set_xlim(max(0.80, float(r["accuracy"].min() - r["accuracy_std"].max() - 0.04)), 1.0)
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
    ax.set_xlabel("Cross-validated accuracy (dot = mean, line = +/- 1 std)")
    ax.set_title("How the six models compare (training flowers only)")
    ax.grid(axis="y", visible=False)
    save(fig, out, "model_comparison.png")
    return res


# --------------------------------------------------------------------------- #
# 5. Evaluation charts
# --------------------------------------------------------------------------- #
def draw_confusion(ax, cm, title):
    cmap = LinearSegmentedColormap.from_list("blues", ["#eaf2fc", "#9ec5f4", "#256abf", "#0d366b"])
    ax.imshow(cm, cmap=cmap, vmin=0, vmax=cm.max())
    ax.grid(False)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(SPECIES)
    ax.set_yticklabels(SPECIES)
    ax.set_xlabel("Predicted species")
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() * 0.5 else INK)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)


def plot_boundary(model, df, name, out):
    cols = ["petal_length", "petal_width"]
    m = clone(model).fit(df[cols], df["species"])
    x0, x1 = max(0, df[cols[0]].min() - 0.4), df[cols[0]].max() + 0.4  # sizes cannot be negative
    y0, y1 = max(0, df[cols[1]].min() - 0.3), df[cols[1]].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
    grid = pd.DataFrame({cols[0]: xx.ravel(), cols[1]: yy.ravel()})
    Z = pd.Series(m.predict(grid)).map(CODE).to_numpy().reshape(xx.shape)
    style()
    fig, ax = plt.subplots(figsize=(6.8, 4.6))
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5], colors=[COLORS[s] for s in SPECIES], alpha=0.14)
    ax.grid(False)
    for sp in SPECIES:
        s = df[df["species"] == sp]
        ax.scatter(s[cols[0]], s[cols[1]], s=26, color=COLORS[sp], marker=MARKERS[sp],
                   edgecolor=SURFACE, linewidth=0.8)
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")
    ax.set_title(f"{name}: decision regions from petal size alone")
    ax.legend(handles=species_legend(), frameon=False, loc="upper left")
    save(fig, out, "decision_boundary.png")


def plot_importance(df, out):
    rf = RandomForestClassifier(n_estimators=500, random_state=SEED).fit(df[FEATURES], df["species"])
    imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
    style()
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.barh([f.replace("_", " ") for f in imp.index], imp.values, color=BLUE, height=0.6)
    for y_, v in enumerate(imp.values):
        ax.text(v + 0.01, y_, f"{v:.0%}", va="center", color=INK2, fontsize=9)
    ax.set_xlim(0, imp.max() + 0.1)
    ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
    ax.set_xlabel("Share of Random Forest importance")
    ax.set_title("Which measurement matters most")
    ax.grid(axis="y", visible=False)
    save(fig, out, "feature_importance.png")
    return imp.sort_values(ascending=False)


# --------------------------------------------------------------------------- #
# 6. Main
# --------------------------------------------------------------------------- #
def describe(model, row):
    x = pd.DataFrame([row], columns=FEATURES)
    pred = model.predict(x)[0]
    proba = dict(zip(model.classes_, model.predict_proba(x)[0]))
    conf = ", ".join(f"{k} {v:.0%}" for k, v in sorted(proba.items(), key=lambda kv: -kv[1]))
    return pred, conf


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", help="path to IRIS[1].csv")
    ap.add_argument("--out", default=OUT_DIR)
    ap.add_argument("--predict", nargs=4, type=float,
                    metavar=("SEPAL_LEN", "SEPAL_WID", "PETAL_LEN", "PETAL_WID"),
                    help="classify one new flower (cm)")
    args, _ = ap.parse_known_args()  # ignores extra arguments Jupyter/Colab passes in
    os.makedirs(args.out, exist_ok=True)

    df = load_data(args.csv)
    explore(df, args.out)
    X, y = df[FEATURES], df["species"]

    # hold-out split: untouched until the final evaluation
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    log(f"\nSplit: {len(X_tr)} flowers for training/model selection, {len(X_te)} held back for the final test.")

    log("\n=== Model comparison: 10 x repeated 5-fold CV on the training flowers ===")
    res = compare_models(X_tr, y_tr, args.out)
    log(res.round(4).to_string(index=False))
    best_name = res.iloc[0]["model"]
    gap = res.iloc[0]["accuracy"] - res.iloc[1]["accuracy"]
    log(f"\nBest on average: {best_name}. Its lead over the runner-up is {gap:.1%}, "
        f"compared with a run-to-run spread of about {res.iloc[0]['accuracy_std']:.1%}; "
        "on a dataset this small the top models are practically tied.")
    best = make_models()[best_name]

    # 1) hold-out test
    fitted = clone(best).fit(X_tr, y_tr)
    pred_te = fitted.predict(X_te)
    acc_te = accuracy_score(y_te, pred_te)
    log(f"\n=== Hold-out test ({len(X_te)} unseen flowers) ===")
    log(f"Accuracy: {acc_te:.1%}  ({int((pred_te == y_te).sum())} of {len(y_te)} correct)")
    log(classification_report(y_te, pred_te, digits=3))
    cm_te = confusion_matrix(y_te, pred_te, labels=SPECIES)

    # 2) cross-validated predictions for every flower
    cv_pred = cross_val_predict(clone(best), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
    acc_cv = accuracy_score(y, cv_pred)
    log(f"=== 5-fold cross-validated predictions for all {len(y)} flowers ===")
    log(f"Accuracy: {acc_cv:.1%}  ({int((cv_pred == y).sum())} of {len(y)} correct)")
    log(classification_report(y, cv_pred, digits=3))
    cm_cv = confusion_matrix(y, cv_pred, labels=SPECIES)
    wrong = df[cv_pred != y.to_numpy()].assign(predicted=cv_pred[cv_pred != y.to_numpy()])
    if len(wrong):
        log("Flowers the model got wrong in cross-validation:")
        log(wrong.round(1).to_string())
        wrong.to_csv(os.path.join(args.out, "misclassified_flowers.csv"), index=False)

    style()
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
    draw_confusion(axes[0], cm_te, f"Hold-out test ({len(y_te)} flowers): {acc_te:.0%}")
    draw_confusion(axes[1], cm_cv, f"5-fold CV, all {len(y)} flowers: {acc_cv:.0%}")
    axes[0].set_ylabel("True species")
    save(fig, args.out, "confusion_matrix.png")

    plot_boundary(best, df, best_name, args.out)
    imp = plot_importance(df, args.out)
    log("Random Forest feature importance: " + ", ".join(f"{k} {v:.0%}" for k, v in imp.items()))

    # final model on all data
    final = clone(best).fit(X, y)
    path = os.path.join(args.out, "iris_model.joblib")
    joblib.dump({"model": final, "features": FEATURES, "classes": list(final.classes_)}, path)
    log(f"\nFinal {best_name} trained on all {len(df)} flowers and saved to {path}")

    log("\nExample predictions with the final model:")
    samples = [(args.predict if args.predict else None, "your flower"),
               ([5.1, 3.5, 1.4, 0.2], "small petals"), ([6.0, 2.9, 4.5, 1.5], "medium petals"),
               ([6.9, 3.1, 5.4, 2.1], "large petals")]
    for row, label in samples:
        if row is None:
            continue
        pred, conf = describe(final, row)
        log(f"  {row} ({label}) -> {pred}   [{conf}]")

    with open(os.path.join(args.out, "results_summary.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(_LOG))
    log(f"\nSaved outputs to {os.path.abspath(args.out)}")


if __name__ == "__main__":
    main()


Loaded 150 flowers from /content/IRIS[1].csv
Missing values dropped: 0.  Exact duplicate rows kept: 3 (real flowers can share measurements).

=== Data overview ===
Flowers per species: setosa 50, versicolor 50, virginica 50

Average measurements (cm) per species:
            sepal_length  sepal_width  petal_length  petal_width
species                                                         
setosa              5.01         3.42          1.46         0.24
versicolor          5.94         2.77          4.26         1.33
virginica           6.59         2.97          5.55         2.03

Petal length range (cm) per species:
            min  max
species             
setosa      1.0  1.9
versicolor  3.0  5.1
virginica   4.5  6.9
-> Setosa is completely separable: its largest petal (1.9 cm) is shorter than the smallest versicolor petal (3.0 cm).
-> Versicolor and virginica overlap by 0.6 cm of petal length, so that pair is where mistakes can happen.

Split: 120 flowers for training/model selec